### Why this exists

litesearch could already ingest a Sanskrit text. It just could not do anything useful with one,
and the failure was silent — three separate layers each assumed a document looks like English
prose or markdown.

**Structure collapsed.** `detect_mode` looks for markdown `#` headings or uppercase-ASCII
structural words (`CHAPTER`, `ADHYAYA`). A Sanskrit source has neither, so it fell through to
`window` mode:

| document | mode | result |
|---|---|---|
| GRETIL Manusmṛti, 306,920 chars, 12 adhyāyas | `window` | **one** node, `Pages 1–1: …` |
| Lalitā Sahasranāma, 17,465 chars, 200 sections | `window` | **one** node |

**Chunks cut mid-verse.** `SafeFastChunker(chunk_size=512)` splits on *bytes*, and Devanagari is
three bytes per character — about 170 characters, roughly two verses, cut wherever it lands.
Measured on the Īśopaniṣad, **2 of 3** internal boundaries fell inside a verse; chunk 0 ended on
the first half of verse 4 and chunk 1 opened with the second half. Neither held a complete verse
or a resolvable citation.

**FTS could not match a Sanskrit query.** Against the tokenizer chain as configured:

| query | against | hits |
|---|---|---|
| `देवी` | `दे॒वी` (accented) | **0** |
| `श्रीमाता` | `śrīmātā` | **0** |
| `kuṇḍa` | `cidagnikuṇḍasambhūtā` | **0** |

A verse is 40–90 characters and already carries a boundary marker and a stable id. This module
takes those seriously: the citation *is* the chunk boundary, the chunk id, and the address that
builds the tree.

In [ ]:
#| default_exp sanskrit

In [ ]:
#| hide
from nbdev.showdoc import *

## Normalisation — making two spellings collide

The trick that makes everything else cheap: rather than transliterate at query time, fold both
scripts to the same lossy ASCII key. `simplify casefold` already turns IAST `śrīmātā` into
`srimata`; `deva2ascii` is chosen so Devanagari `श्रीमाता` lands on `srimata` too. They meet in
the middle and no transliteration library is needed on the hot path.

Vedic tone marks are stripped first. They are not decoration — the corpus this was built against
carries 1,644 of them — and no one types them into a search box.

In [ ]:
#| export
import re, unicodedata
from fastcore.all import L, Path, ifnone

# Vedic svara/tone marks. `porter simplify casefold` folds Latin diacritics but leaves these, so
# an unaccented query finds nothing in an accented text — measured 0 hits for `देवी` against `दे॒वी`.
# Latin combining marks are deliberately *not* here: this normalises Devanagari, and folding IAST
# is `_latn_fold`'s job. Nor are the candrabindu signs, which carry a nasal and become `m`.
VEDIC_MARKS = re.compile('['
    '॑-॔'      # udatta, anudatta, devanagari grave/acute
    '᳐-᳹'      # vedic extensions: tone marks, nasalisations
    '꣠-꣰'      # devanagari extended combining digits, stopping before the candrabindu block
    ']')

DANDA, DDANDA = '।', '॥'          # । ॥

# Devanagari -> ASCII. Deliberately lossy and deliberately *matched to what `simplify casefold`
# does to IAST*: `ś`->`s`, `ī`->`i`, so `श्रीमाता` and `śrīmātā` both fold to `srimata` and meet in
# the middle. That is what makes cross-script search work without transliterating at query time.
_INDEP = {'अ':'a','आ':'a','इ':'i','ई':'i','उ':'u','ऊ':'u','ऋ':'r','ॠ':'r','ऌ':'l','ॡ':'l',
          'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऍ':'e','ऎ':'e','ऑ':'o','ऒ':'o'}
_CONS  = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'n','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'n',
          'ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
          'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v',
          'श':'s','ष':'s','स':'s','ह':'h','ळ':'l','ऴ':'l','ऱ':'r','ऩ':'n',
          'क़':'k','ख़':'kh','ग़':'g','ज़':'z','ड़':'d','ढ़':'dh','फ़':'f','य़':'y'}
_MATRA = {'ा':'a','ि':'i','ी':'i','ु':'u','ू':'u','ृ':'r','ॄ':'r','ॢ':'l','ॣ':'l',
          'े':'e','ै':'ai','ो':'o','ौ':'au','ॅ':'e','ॆ':'e','ॉ':'o','ॊ':'o'}
# `ꣳ`/`ꣲ`/`꣰` are the Vedic candrabindu signs — a nasal, not an accent, so they fold to `m` and
# line up with the `ṃ` an IAST edition of the same line prints.
_SIGN  = {'ं':'m','ः':'h','ँ':'m','ऽ':'', '़':'', 'ꣲ':'m','ꣳ':'m','ꣴ':'m','ꣵ':'m','ꣶ':'m','ꣷ':'m'}
_VIRAMA = '्'
_DIGITS = {chr(0x966+i): str(i) for i in range(10)}

def strip_vedic(s:str) -> str:
    'Drop Vedic tone marks from Devanagari, leaving Latin diacritics alone. NFC out.'
    if not s: return ''
    return unicodedata.normalize('NFC', VEDIC_MARKS.sub('', unicodedata.normalize('NFD', s)))

def deva2ascii(s:str) -> str:
    """Devanagari to a bare ASCII key, applying the implicit `a` of an unmarked consonant.

    Not a transliteration scheme anyone should read — it is an *index key*, chosen so that the
    Devanagari and the IAST spelling of a word collide."""
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _CONS:
            out.append(_CONS[ch]); i += 1
            # look past a nukta that was not pre-composed
            while i < n and s[i] == '़': i += 1
            if i < n and s[i] == _VIRAMA: i += 1; continue          # bare consonant, no vowel
            if i < n and s[i] in _MATRA: out.append(_MATRA[s[i]]); i += 1; continue
            out.append('a'); continue                                # implicit vowel
        if ch in _INDEP: out.append(_INDEP[ch]); i += 1; continue
        if ch in _SIGN:  out.append(_SIGN[ch]);  i += 1; continue
        if ch in _MATRA: out.append(_MATRA[ch]); i += 1; continue    # orphan matra
        if ch in _DIGITS: out.append(_DIGITS[ch]); i += 1; continue
        if ch == _VIRAMA: i += 1; continue
        out.append(ch); i += 1
    return ''.join(out)

def _latn_fold(s:str) -> str:
    'IAST/ISO to ASCII — the same folding `simplify casefold` already applies to a Latin token.'
    d = unicodedata.normalize('NFD', s)
    return ''.join(c for c in d if unicodedata.category(c) != 'Mn').lower()

_DEVA = re.compile('[ऀ-ॿ᳐-᳹꣠-ꣿ]')

def detect_script(s:str) -> str:
    "`'deva'`, `'latn'` or `'other'` — enough to pick a folding path."
    if not s: return 'other'
    d = len(_DEVA.findall(s))
    l = sum(1 for c in s if 'a' <= c.lower() <= 'z' or unicodedata.category(c) == 'Ll')
    if d and d >= l: return 'deva'
    return 'latn' if l else 'other'

def fold_token(s:str) -> str:
    'The ASCII index key for one token, whatever script it arrived in.'
    s = strip_vedic(s)
    return _latn_fold(deva2ascii(s) if _DEVA.search(s) else s)

In [ ]:
#| hide
# the whole point: the same word in either script folds to one key
for deva, iast in [('श्रीमाता','śrīmātā'), ('देवी','devī'), ('दे॒वी','devī'),
                   ('चिदग्निकुण्डसम्भूता','cidagnikuṇḍasambhūtā'), ('गणपतिꣳ','gaṇapatiṃ')]:
    assert fold_token(deva) == fold_token(iast), (deva, fold_token(deva), iast, fold_token(iast))
assert fold_token('श्रीमाता') == 'srimata'
# accents fold away; they are the difference between 0 hits and 1
assert fold_token('दे॒वी') == fold_token('देवी') == 'devi'
assert strip_vedic('सर॑स्वती॒') == 'सरस्वती'
# strip_vedic must leave Latin diacritics alone — folding those is _latn_fold's job
assert strip_vedic('śrīmātā') == 'śrīmātā'
assert detect_script('श्रीमाता') == 'deva' and detect_script('śrīmātā') == 'latn'
assert detect_script('') == 'other'
# the implicit `a` of an unmarked consonant, and virama suppressing it
assert deva2ascii('क') == 'ka' and deva2ascii('क्') == 'k' and deva2ascii('का') == 'ka'

## An FTS5 tokenizer that folds Sanskrit

The fold could have been a second indexed column, the way an application usually does it. A
tokenizer is better: `content` stays exactly as ingested, nothing has to stay in sync, and FTS5
already has the right mechanism — **colocated tokens**, its synonym feature.

It *wraps* rather than replaces, so it composes with the chain already in use. Measured against
the bare chain it is strictly additive — `running`, `cat`, `fts_search` and `quickli` all behave
identically, while `srimata` against `श्रीमाता` goes from 0 hits to 2. That is why it is the
default for every store rather than something a Sanskrit corpus opts into: a store's tokenizer is
fixed when its table is created, and the first document ingested should not get to decide it.

In [ ]:
#| export
SANSKRIT_TOKENIZE = 'sanskrit'

def sanskrit_tokenizer(con, args):
    """A *wrapping* FTS5 tokenizer that folds Sanskrit so a query matches regardless of script.

    Emits the ASCII fold of each token as a **colocated** token — FTS5's synonym mechanism — so
    `content` is stored exactly as ingested while `देवी`, `दे॒वी`, `devī` and `devi` all reach the
    same row. Nothing transliterates at query time: both scripts fold to ASCII through fixed
    tables, which is what keeps this cheap enough to sit in the index path.

    It wraps rather than replaces so it can sit *inside* the existing chain —
    `porter simplify casefold 1 sanskrit unicodewords` still stems English and still keeps
    identifiers whole, and only adds tokens on top. Being purely additive is what makes it safe as
    a default for every store rather than something a Sanskrit corpus has to opt into.

    **Inside, not outside, and the difference is a silent loss of hits.** Wrapping the whole chain
    computes the fold on porter's output, and porter never touches Devanagari: `धर्मक्षेत्रे` is
    then indexed under the unstemmed `dharmaksetre` while the ASCII query for the same word is
    stemmed to `dharmaksetr`, and the two never meet. Emitted from inside, the fold is just another
    token porter goes on to stem, so both sides land on the same string. Sanskrit case endings make
    this routine rather than rare — the locative `-e` alone accounts for most of it."""
    import apsw.fts5
    rest = [a for a in args if '=' not in a]
    opts = dict(a.split('=', 1) for a in args if '=' in a)
    keep_orig = opts.get('original', '1') != '0'
    inner = con.fts5_tokenizer(rest[0], rest[1:]) if rest else apsw.fts5.UnicodeWordsTokenizer(con, [])
    def tok(utf8, flags, locale):
        for start, end, *toks in inner(utf8, flags, locale):
            out = []
            for t in toks:
                if keep_orig and t not in out: out.append(t)
                if (f := fold_token(t)) and f not in out: out.append(f)
            if out: yield (start, end, *out)
    return tok

def register_sanskrit(db):
    "Register the `sanskrit` FTS5 tokenizer on a connection. Idempotent."
    conn = getattr(db, 'conn', db)
    try: conn.register_fts5_tokenizer(SANSKRIT_TOKENIZE, sanskrit_tokenizer)
    except Exception: pass
    return db

In [ ]:
#| hide
import apsw, apsw.fts5
from litesearch.core import _FTS_TOKENIZE
_c = apsw.Connection(':memory:')
apsw.fts5.register_tokenizers(_c, apsw.fts5.map_tokenizers); register_sanskrit(_c)
_c.execute(f"CREATE VIRTUAL TABLE _t USING fts5(c, tokenize='{_FTS_TOKENIZE}')")
for _r in ['श्रीमाता चिदग्नि-कुण्ड-सम्भूता दे॒वी', 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः',
           'The cats were running and fts_search stayed whole']:
    _c.execute('INSERT INTO _t(c) VALUES(?)', (_r,))
def _hits(q): return _c.execute('SELECT count(*) FROM _t WHERE _t MATCH ?', (q,)).fetchone()[0]
# cross-script and accent-blind
assert _hits('श्रीमाता') and _hits('srimata') and _hits('śrīmātā') and _hits('देवी')
# and English is untouched: porter still stems, identifiers still survive
assert _hits('running') and _hits('cat') and _hits('fts_search')

# Regression: the fold has to be emitted *inside* porter. With `sanskrit` wrapping the chain the
# fold was computed on porter's output — Devanagari passes porter untouched, so the index held the
# unstemmed `dharmaksetre` while the query for the same word stemmed to `dharmaksetr` and missed.
# `srimata` is unaffected (porter leaves it alone), which is exactly why the old test passed.
# These are locatives in `-e`, the ending that makes the mismatch routine in Sanskrit.
for _q in ['dharmaksetre', 'kuruksetre', 'dharmakṣetre']:
    assert _hits(_q) == 1, f'{_q!r} should reach the Devanagari row without a wildcard'
# and the exact, unwildcarded query is what has to work: a prefix search hid this bug
assert _hits('"dharmaksetre"') == 1

## Citations, and where a verse ends

Three boundary markers, tried in that order because that is their order of reliability:

| source | marker |
|---|---|
| GRETIL plain text | `// Mn_1.1 //`, `\|\| IsUp_4 \|\|`, `\|\| BrhUp_1,1.2 \|\|` |
| Devanagari editions | `॥ १॥`, then a bare `॥` closing a line |
| GRETIL TEI, SARIT | `<lg xml:id="Manu_1.1">` |

A unit always *ends with* its marker, so a chunk is never a fragment whose citation lives in the
next one. Two details cost real bugs to find. A `॥` also *opens* a bracketed title in Devanagari,
so only a `॥` that closes a line may cut. And a translation printed under a verse sits after its
daṇḍa, so a naive cut files it against the following verse.

In [ ]:
#| export
# `// Mn_1.1 //`, `|| IsUp_4 ||`, `|| BrhUp_1,1.2 ||` — GRETIL's citation marker, which is also the
# verse boundary and the verse's stable id.
# The bracketed tail matters: GRETIL marks a verse that is numbered differently in another edition
# as `// Mn_3.67[57M] //`. Without it those citations go unrecognised and every verse between two
# *recognised* ones packs into a single unit — measured one 25,974-character chunk in Manusmṛti.
# The separator is `_` in GRETIL (`Mn_1.1`) and `.` in DCS (`R.1.1.1`); the siglum is letters only,
# which keeps the number from being swallowed into it by backtracking.
CITE_RE = re.compile(r'(?:\|\||//)\s*([A-Za-zĀ-ſ]+)[_\s.]([\d,.\-]+[a-z]?(?:\[[^\]\n]{0,16}\])?)\s*(?:\|\||//)')
# A Devanagari verse number: `॥ १॥` or `॥ 12 ॥`
_DEVNUM = re.compile(r'॥\s*([\d०-९]+)\s*॥')

def cite_parts(c:str) -> tuple:
    """`'BrhUp_1,1.2'` -> `('BrhUp', ['1','1','2'])` — the siglum and its hierarchy.

    Handles both separators in the wild: GRETIL writes `Mn_1.1`, DCS writes `R.1.1.1`. The
    hierarchy is what `build_tree` turns into adhyāya/chapter nodes, so a corpus with no markup at
    all still gets a navigable tree."""
    c = re.sub(r'\[[^\]]*\]', '', (c or '')).strip()   # drop a `[57M]` alternate-edition number
    m = re.match(r'([^\s_]+)[_\s]([\d,.\-]+[a-z]?)$', c) or re.match(r'([A-Za-zĀ-ſ]+)[.]([\d,.\-]+[a-z]?)$', c)
    if not m: return (c, [])
    return m.group(1), [p for p in re.split(r'[,.]', m.group(2)) if p]

_GLOSS = re.compile(r'(?:[ \t]*\n)?[ \t]*>[^\n]*')
_HEAD  = re.compile(r'^[ \t]*#{1,6}[ \t]+\S', re.M)

def _with_gloss(text:str, end:int) -> int:
    """Extend a cut past the gloss lines that trail it.

    A translation printed under a verse sits *after* its daṇḍa, so a naive cut files it against the
    following verse — the meaning of verse 1 retrieved as if it explained verse 2. Everything a
    `>` line says belongs to the verse above it."""
    while (m := _GLOSS.match(text, end)) and m.end() > end: end = m.end()
    return end

def verse_spans(text:str) -> L:
    """Split `text` into verse units, returning `(start, end, text, citation)`.

    Boundaries are tried in order — citation marker, then a numbered `॥ १॥`, then a bare `॥` —
    because that is the order of reliability in real sources. A unit always *ends with* its marker,
    so a chunk is never a fragment whose citation lives in the next chunk."""
    if not (text or '').strip(): return L()
    cuts = []
    for m in CITE_RE.finditer(text): cuts.append((_with_gloss(text, m.end()), f'{m.group(1)}_{m.group(2)}'))
    if not cuts:
        for m in _DEVNUM.finditer(text): cuts.append((_with_gloss(text, m.end()), m.group(1)))
    if not cuts:
        # Only a double daṇḍa that *closes* a line. Devanagari brackets a title or a section as
        # `॥ text ॥`, so cutting after every `॥` splits the opening bracket off as a chunk of one
        # character. Both conventions are here: a Devanagari edition prints `॥`, and an IAST one
        # prints `||` or `//`. Without the ASCII pair, a romanised text that carries no citations —
        # which is most of a stotra collection — got no verse boundaries at all, and the whole
        # section arrived as a single unit whose metre scans as one 31-syllable nothing.
        for m in re.finditer(r'(?:॥|\|\||//)[ \t]*(?=\n|$)', text):
            cuts.append((_with_gloss(text, m.end()), None))
    # A heading always opens a new unit. Without this a stotra whose sections are separated by
    # headings rather than by numbering packs the whole text into a handful of chunks — measured
    # 48 chunks for a 318-section namāvali before this line, 318 after.
    cuts += [(m.start(), None) for m in _HEAD.finditer(text) if m.start() > 0]
    cuts.sort()
    out, prev = L(), 0
    for end, cite in cuts:
        if end <= prev: continue
        if (seg := text[prev:end]).strip(): out.append((prev, end, seg.strip(), cite))
        prev = end
    if (tail := text[prev:]).strip(): out.append((prev, len(text), tail.strip(), None))
    return out

_CLAUSE = re.compile(r'(?:॥|।|//|/|\|)[ \t]*')

def _clause_spans(seg:str) -> L:
    """Split an over-long unit at its clause daṇḍas, never mid-clause.

    Both daṇḍa conventions are here because GRETIL writes IAST verse with ASCII `/` and `//` while
    Devanagari sources use `।` and `॥`."""
    parts, prev = L(), 0
    for m in _CLAUSE.finditer(seg):
        if (s := seg[prev:m.end()]).strip(): parts.append(s.strip())
        prev = m.end()
    if (t := seg[prev:]).strip(): parts.append(t.strip())
    return parts or L([seg.strip()])

def _atoms(seg:str, max_chars:int) -> L:
    """The smallest pieces an over-long unit may be cut into, each with its gloss attached.

    Three passes, because one is not enough on real text. Daṇḍa first; then line breaks for
    anything still oversized, since accented Vedic recitation carries no daṇḍa inside a section and
    a daṇḍa split alone left a 2,282-character piece in Rudram; then gloss lines are folded back
    into the piece above them, which is what makes an atom safe to pack without ever separating a
    translation from its verse. An atom longer than `max_chars` is emitted whole — a clause with no
    interior boundary is not something to cut blind."""
    out = L()
    for p in _clause_spans(seg):
        if len(p) <= max_chars: out.append(p); continue
        lines = [l.strip() for l in p.splitlines() if l.strip()]
        out += (lines if len(lines) > 1 else [p])
    atoms = L()
    for p in out:
        if p.lstrip().startswith('>') and atoms: atoms[-1] += '\n' + p
        else: atoms.append(p)
    return atoms

In [ ]:
#| hide
assert cite_parts('BrhUp_1,1.2') == ('BrhUp', ['1','1','2'])
assert cite_parts('Mn_1.1') == ('Mn', ['1','1'])
assert cite_parts('R.1.1.1') == ('R', ['1','1','1'])            # DCS separates with a dot
assert cite_parts('Mn_3.67[57M]') == ('Mn', ['3','67'])         # alternate-edition number dropped
# a unit ends with its marker, and the gloss under a verse belongs to that verse
_sp = verse_spans('a b || X_1 ||\n> meaning of one\nc d || X_2 ||')
assert len(_sp) == 2, _sp
assert _sp[0][3] == 'X_1' and 'meaning of one' in _sp[0][2]
assert 'meaning of one' not in _sp[1][2]
# `॥` opening a bracketed title must not cut
assert len(verse_spans('॥ शीर्षकम् ॥\n')) == 1

## Chunkers

`VerseChunker` is chonkie-shaped, so it drops straight into `add_doc(chunker=...)` and
`chunk_markdown(text, chunker)` — the tree, the store, the ANN index and `read()` are unchanged.

Two guards, both load-bearing. **Packing** exists because a namāvali line is 18 characters and one
line is not a retrievable chunk; it never packs across a citation or a heading, so a chunk still
maps to one citable range. **Splitting** exists because Sanskrit prose — the Upaniṣads run ten
lines to a single `|| ref ||` — would otherwise produce a chunk far larger than the embedder can
represent. `ProseChunker` is the same machinery on a larger budget, for commentary and bhāṣya.

In [ ]:
#| export
VERSE_TARGET, VERSE_MAX = 300, 900

def _width(parts) -> int:
    'Length the parts will have once joined — the newlines count against the budget too.'
    return sum(map(len, parts)) + max(len(parts) - 1, 0)

def chunk_verses(text:str,                 # source text
                 target:int=VERSE_TARGET,  # pack short units forward up to this many chars
                 max_chars:int=VERSE_MAX,  # split a unit longer than this at clause boundaries
                 pack_cited:bool=False,    # let consecutive cited units share a chunk up to `target`
                 ) -> L:
    """Verse-aware chunks: cut on citation/daṇḍa, pack short units, split long prose.

    Two guards, and both are load-bearing. Packing exists because a namāvali line is 18 characters
    and one line is not a retrievable chunk; it never packs *across* a heading, so a chunk still
    maps to a contiguous, citable range. Splitting exists because Sanskrit prose — the Upaniṣads
    run ten lines to one `|| ref ||` — would otherwise produce a chunk far larger than anything the
    embedder can represent.

    `pack_cited` is what makes a larger `target` mean anything on a citation-dense text. By default
    a citation closes the chunk immediately, so on GRETIL — where *every* unit ends in one — the
    target is never consulted and `ProseChunker`'s larger budget was inert: measured 10 chunks for
    the Īśopaniṣad either way. With it on, consecutive cited units pack forward to `target`, which
    still yields one contiguous citable *range* (`IsUp_1–3`) rather than a single address. That is
    the right trade for commentary, where the unit of meaning spans several verses, and the wrong
    one for a verse index, which is why it is off unless `ProseChunker` asks for it."""
    units = verse_spans(text)
    if not units: return L([text.strip()]) if (text or '').strip() else L()
    out, buf, cited = L(), [], False
    def flush():
        nonlocal buf, cited
        if buf: out.append('\n'.join(buf).strip())
        buf, cited = [], False
    for _, _, seg, cite in units:
        # A heading opens a section; packing across it would put the tail of one section and the
        # head of the next into a chunk whose breadcrumb can only name one of them.
        if _HEAD.match(seg): flush()
        if len(seg) > max_chars:
            flush()
            cur = []
            for a in _atoms(seg, max_chars):
                if cur and _width(cur) + len(a) > max_chars: out.append('\n'.join(cur)); cur = []
                cur.append(a)
            if cur: out.append('\n'.join(cur))
            continue
        # close the run *before* it overflows, so packing can never produce a chunk larger than a
        # unit the embedder was never going to fit
        if buf and _width(buf) + len(seg) > max_chars: flush()
        buf.append(seg)
        if cite: cited = True
        # a citation closes the unit; otherwise pack forward until the target is reached
        if (cited and not pack_cited) or _width(buf) >= target: flush()
    flush()
    return out.filter(lambda s: s.strip())


class _Chunk:
    'Minimal chonkie-compatible chunk (chonkie.types.Chunk is imported lazily to keep this light).'
    def __init__(self, text, start_index, end_index, token_count=0):
        self.text, self.start_index, self.end_index, self.token_count = text, start_index, end_index, token_count
    def __repr__(self): return f'_Chunk({self.text[:40]!r}…)'


def _mk_chunks(texts, src):
    'Wrap chunk strings as chonkie chunks, carrying real offsets into `src` where they can be found.'
    try: from chonkie.types import Chunk
    except Exception: Chunk = _Chunk
    out, pos = [], 0
    for t in texts:
        i = src.find(t[:60], pos) if t else -1
        st = i if i >= 0 else pos
        out.append(Chunk(text=t, start_index=st, end_index=st+len(t), token_count=0))
        pos = st + len(t)
    return out


class VerseChunker:
    """A chonkie-shaped chunker whose cuts land on verse boundaries.

    Drop-in for `add_doc(chunker=...)` and `chunk_markdown(text, chunker)`, which is the whole
    point: the tree, the store, the ANN index and `read()` are unchanged."""
    def __init__(self, target:int=VERSE_TARGET, max_chars:int=VERSE_MAX, pack_cited:bool=False):
        self.target, self.max_chars, self.chunk_size = target, max_chars, max_chars
        self.pack_cited = pack_cited
    def chunk(self, text:str):
        return _mk_chunks(chunk_verses(text, self.target, self.max_chars, self.pack_cited), text or '')
    def __call__(self, text:str): return self.chunk(text)
    def __repr__(self):
        return f'{type(self).__name__}(target={self.target}, max_chars={self.max_chars}, pack_cited={self.pack_cited})'


class ProseChunker(VerseChunker):
    """`VerseChunker` tuned for Sanskrit prose — commentary, bhāṣya, the prose Upaniṣads.

    Same boundaries, different budget: prose has no metrical unit to preserve, so packing to a
    larger target beats emitting one chunk per clause. It sets `pack_cited=True`, without which
    the larger target is unreachable on any text that cites every unit — which is most of GRETIL,
    and was the whole corpus this chunker exists for."""
    def __init__(self, target:int=700, max_chars:int=1400, pack_cited:bool=True):
        super().__init__(target, max_chars, pack_cited)

In [ ]:
#| hide
_v = ('īśā vāsyam idaṃ sarvaṃ | tena tyaktena bhuñjīthāḥ || IsUp_1 ||\n'
      'kurvann eveha karmāṇi | evaṃ tvayi nānyatheto || IsUp_2 ||\n'
      'asuryā nāma te lokā | tāṃste pretyābhigacchanti || IsUp_3 ||')
_cs = chunk_verses(_v)
assert len(_cs) == 3, _cs                                    # one verse per chunk
# every internal boundary lands on a marker, and no citation is split across two chunks
import re as _re
assert all(_re.search(r'(॥|।|\|\||//)\s*$', c.rstrip()) for c in _cs)
_seen = [m.group(0) for c in _cs for m in CITE_RE.finditer(c)]
assert len(_seen) == len(set(_seen)) == 3
# round-trip: chunking loses no text
assert ''.join(_v.split()) == ''.join(''.join(_cs).split())
# short units pack forward rather than becoming chunks of their own
assert len(chunk_verses('नमः ॥\n' * 12)) < 12
# and an over-long prose unit is split, at clause boundaries
_long = ' | '.join(['ayam eva ātmā brahma iti vijñāyate'] * 60) + ' || BrhUp_2,5.19 ||'
assert max(map(len, chunk_verses(_long))) <= VERSE_MAX
assert VerseChunker().chunk('') == [] and chunk_verses('') == []
assert isinstance(ProseChunker().max_chars, int) and ProseChunker().max_chars > VerseChunker().max_chars

In [ ]:
#| hide
# A citation closes a chunk, so on a text that cites *every* unit the target was never consulted
# and `ProseChunker`'s larger budget was inert — measured 10 chunks for the Īśopaniṣad either way.
_cited = ''.join(f'short line {i} || X_{i} ||\n' for i in range(8))
assert len(chunk_verses(_cited)) == 8                              # verse: one chunk per citation
assert len(chunk_verses(_cited, 700, 1400, pack_cited=True)) < 8   # prose: packs to the target
assert len(ProseChunker().chunk(_cited)) < len(VerseChunker().chunk(_cited))
assert VerseChunker().pack_cited is False and ProseChunker().pack_cited is True
# packing must not lose text, and a heading must still open a chunk
_txt = 'a || X_1 ||\nb || X_2 ||\nc || X_3 ||'
assert ''.join(_txt.split()) == ''.join(''.join(chunk_verses(_txt, 700, 1400, True)).split())
assert len(chunk_verses('# A\na || X_1 ||\n# B\nb || X_2 ||', 700, 1400, True)) == 2


## Metre — the gaṇas

A Sanskrit verse is built from **gaṇas**: triples of syllable weights, where a syllable is heavy
(*guru*) or light (*laghu*). Eight triples exhaust the possibilities, and every classical metre is
a recipe in them — mandākrāntā is `ma bha na ta ta ga ga`, and a verse either fits that or it does
not. That makes metre one of the few facets of a Sanskrit corpus that is **exactly computable**:
no model, no lexicon, no annotation, just the text.

It earns its place in the index for three reasons. It is a **facet** — *show me the
śārdūlavikrīḍita verses* — which is not otherwise expressible. It is a **fingerprint**: a change of
metre marks a change of register, a quotation, or a different hand, so comparing which metres two
texts use is a real way to read them against each other. And the **gaṇa signature itself** is a
join key, letting verses that share a shape be compared across works that share no vocabulary.

The traditional mnemonic encodes the whole table:

> **यमाताराजभानसलगम्** — *ya-mā-tā-rā-ja-bhā-na-sa-la-gam*

Read three syllables from any starting point and you have that gaṇa's own pattern: `ya` opens
`ya-mā-tā` = ˘ ¯ ¯, and `ya` *is* ˘ ¯ ¯. The test below checks `GANAS` against the mnemonic rather
than trusting the table, because a mistyped triple is a bug nobody ever finds by reading.

### Two systems, not one

The table above is a **varṇa** metre: it fixes the weight of every syllable in a pāda, so a verse
in mandākrāntā always has 17 syllables in a known order. The older **mātrā** system counts morae
instead — a laghu is one, a guru is two — and fixes only how many, leaving the poet to fill them
with whatever syllables they like. A syllable count therefore identifies nothing there: the four
consecutive āryā verses tested below run **32, 35, 41 and 37** syllables.

The unit of the āryā family is the *caturmātra* gaṇa, four morae, of which exactly five shapes
exist. A half is a fixed ladder of them, and the five metres are the ways of pairing two halves:

| metre | first half | second half | morae |
|---|---|---|---|
| āryā | 30 | 27 | 57 |
| gīti | 30 | 30 | 60 |
| upagīti | 27 | 27 | 54 |
| udgīti | 27 | 30 | 57 |
| āryāgīti | 32 | 32 | 64 |

Counting alone would match almost anything, so two constraints do the discriminating: the **odd**
gaṇas (1st, 3rd, 5th, 7th) may not be the ja-gaṇa `˘ ¯ ˘`, and the **6th** must be either that same
ja-gaṇa or four laghus. Both hold in the Sāṃkhyakārikā, where `lgl` turns up constantly in even
positions and never once in an odd one — and with them, 5,000 random weight strings of exactly 57
morae yield zero false āryās.

What this deliberately does not do is classify verse against prose. A syllable count that happens
to divide by four is not evidence of metre, and treating it as such files bhāṣya as śloka.

Still uncovered: the **vaitālīya** family (vaitālīya, aupacchandasika), which counts morae at the
start of a pāda and syllables at the end. Sources reachable from here disagree on the direction of
its closing cadence, and a guessed cadence is worse than a documented gap.

In [ ]:
#| export
from fastcore.all import AttrDict, patch
from fastlite import Database

# --- Devanagari -> IAST, with vowel length kept -------------------------------------------------
# `deva2ascii` deliberately destroys vowel length: that is exactly what makes `श्रीमाता` and
# `srimata` collide in the index. Scansion needs precisely what the fold throws away, since a
# syllable is heavy *because* its vowel is long — reuse the fold here and every syllable comes back
# light and no verse scans at all. So metre gets its own transliteration and the two never share a
# table.
_IV = {'अ':'a','आ':'ā','इ':'i','ई':'ī','उ':'u','ऊ':'ū','ऋ':'ṛ','ॠ':'ṝ','ऌ':'ḷ','ॡ':'ḹ',
       'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऍ':'e','ऎ':'e','ऑ':'o','ऒ':'o'}
_MV = {'ा':'ā','ि':'i','ी':'ī','ु':'u','ू':'ū','ृ':'ṛ','ॄ':'ṝ','ॢ':'ḷ','ॣ':'ḹ',
       'े':'e','ै':'ai','ो':'o','ौ':'au','ॅ':'e','ॆ':'e','ॉ':'o','ॊ':'o'}
_CI = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'ṅ','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'ñ',
       'ट':'ṭ','ठ':'ṭh','ड':'ḍ','ढ':'ḍh','ण':'ṇ','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
       'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v',
       'श':'ś','ष':'ṣ','स':'s','ह':'h','ळ':'ḻ','ऴ':'ḻ','ऱ':'r','ऩ':'n',
       'क़':'k','ख़':'kh','ग़':'g','ज़':'j','ड़':'ḍ','ढ़':'ḍh','फ़':'ph','य़':'y'}
# Anusvāra and visarga are codas — they close a syllable and make it heavy — so they must survive
# here even though the fold turns them into `m` and `h`. The avagraha marks a vowel that was
# *elided*: it is not pronounced and does not scan, so it goes.
_SI = {'ं':'ṃ','ः':'ḥ','ँ':'ṃ','ऽ':'','़':'','ॐ':'oṃ'}

def deva2iast(s:str) -> str:
    'Devanagari to IAST, keeping the vowel length that scansion depends on.'
    s = strip_vedic(s or '')
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _CI:
            out.append(_CI[ch]); i += 1
            while i < n and s[i] == '़': i += 1
            if i < n and s[i] == _VIRAMA: i += 1; continue           # bare consonant, no vowel
            if i < n and s[i] in _MV: out.append(_MV[s[i]]); i += 1; continue
            out.append('a'); continue                                # the implicit vowel
        if ch in _IV: out.append(_IV[ch]); i += 1; continue
        if ch in _SI: out.append(_SI[ch]); i += 1; continue
        if ch in _MV: out.append(_MV[ch]); i += 1; continue           # orphan mātrā
        if ch in _DIGITS: out.append(' '); i += 1; continue           # a verse number is not a syllable
        if ch == _VIRAMA: i += 1; continue
        out.append(ch); i += 1
    return ''.join(out)

# --- syllables and their weights ----------------------------------------------------------------
# `ai` and `au` are one vowel each, and the aspirates are one *consonant* each — `atha` is
# light-light where `artha` is heavy-light. Both tables are therefore matched longest-first.
_VOW2 = ('ai', 'au')
_VOW1 = frozenset('aāiīuūṛṝḷḹeo')
_LONG = frozenset(('ā','ī','ū','ṝ','ḹ','e','o','ai','au'))    # `e` and `o` are always long
_ASP  = ('kh','gh','ch','jh','ṭh','ḍh','th','dh','ph','bh')
_CON1 = frozenset('kgṅcjñṭḍṇtdnpbmyrlvśṣshḻ')
_CODA = frozenset('ṃṁḥ')
_WORDS = re.compile(r'[^\W\d_]+')

def _phones(text:str) -> list:
    """`[('V'|'C', unit)]` for a Sanskrit string, matched **word by word**.

    The word boundary is load-bearing, and it is the thing a naive scanner gets wrong. Matching
    vowels longest-first across the whole string fuses `a` and `i` over a gap: `jīva iti` comes out
    as `jī-vai-ti`, three syllables where there are four, and one missing syllable is enough to make
    the pāda the wrong length and the verse match no metre at all.

    Consonant clusters are the opposite case and *do* count across a boundary — `tat sarvam` closes
    its first syllable on `t`+`s` — so the per-word units are concatenated back into one stream and
    only the vowel matching is fenced. Hyphens fence too: GRETIL writes compounds with them, and
    they are orthography rather than phonology."""
    s = deva2iast(text) if _DEVA.search(text or '') else (text or '')
    out = []
    for w in _WORDS.findall(s.lower()):
        i, n = 0, len(w)
        while i < n:
            if w[i:i+2] in _VOW2: out.append(('V', w[i:i+2])); i += 2; continue
            if w[i:i+2] in _ASP:  out.append(('C', w[i:i+2])); i += 2; continue
            if w[i] in _VOW1:     out.append(('V', w[i]));     i += 1; continue
            if w[i] in _CON1 or w[i] in _CODA: out.append(('C', w[i])); i += 1; continue
            i += 1                                          # not a phoneme; skip it
    return out

def syllables(text:str) -> L:
    """The syllables of a Sanskrit string as `(syllable, guru)` pairs.

    A syllable is heavy (*guru*) when its vowel is long, when it is closed by anusvāra or visarga,
    or when **two or more** consonants follow before the next vowel. Two is the threshold because a
    single consonant between two vowels is the *onset* of the following syllable, not a coda on the
    preceding one — `atha` is light-light, `artha` is heavy-light.

    The exception is the end of the text, where a trailing consonant has no following vowel to be
    the onset of and therefore does close its syllable: `gam` is guru even though only one consonant
    follows the `a`. That is what makes the last syllable of `yamātārājabhānasalagam` the `ga` the
    mnemonic says it is, and it is the difference between the gaṇa table checking out and not."""
    ph = _phones(text)
    out, onset, coda_at = L(), '', -1
    for j, (k, u) in enumerate(ph):
        # a phone already spent as the previous syllable's coda is not also this one's onset —
        # without the check `kaṃsa` reads back as `kaṃ`+`ṃsa` and `duḥkha` as `duḥ`+`ḥkha`
        if k == 'C':
            if j != coda_at: onset += u
            continue
        run, closes = [], True
        for k2, u2 in ph[j+1:]:
            if k2 == 'V': closes = False; break
            run.append(u2)
        heavy = (u in _LONG or any(c in _CODA for c in run) or len(run) >= 2
                 or (closes and len(run) >= 1))
        cod = ''.join(c for c in run[:1] if c in _CODA)
        if cod: coda_at = j + 1
        out.append((onset + u + cod, heavy))
        onset = ''
    return out

def scan(text:str) -> str:
    'The guru/laghu skeleton of a string as `g`/`l` — what every metre is matched against.'
    return ''.join('g' if h else 'l' for _, h in syllables(text))

# --- the gaṇas ----------------------------------------------------------------------------------
# The eight gaṇas, plus the two single-syllable fillers a recipe uses to make up an odd length.
# Checked against the `yamātārājabhānasalagam` mnemonic in the tests rather than trusted.
GANAS = {'ya':'lgg', 'ma':'ggg', 'ta':'ggl', 'ra':'glg',
         'ja':'lgl', 'bha':'gll', 'na':'lll', 'sa':'llg',
         'ga':'g',   'la':'l'}
_OF_GANA = {v: k for k, v in GANAS.items() if len(v) == 3}

def ganas(pattern:str) -> L:
    """The gaṇa names of a `g`/`l` pattern, three syllables at a time.

    A trailing one or two syllables are named `ga`/`la`, which is how a handbook writes a recipe:
    indravajrā is `ta ta ja ga ga` — eleven syllables as 3+3+3+1+1."""
    s, out = pattern or '', L()
    cut = len(s) - len(s) % 3
    out += L(_OF_GANA[s[i:i+3]] for i in range(0, cut, 3))
    return out + L('ga' if c == 'g' else 'la' for c in s[cut:])

def gana_pattern(recipe:str) -> str:
    'The `g`/`l` pattern a gaṇa recipe spells out: `ta ta ja ga ga` -> `gglgglglgg`.'
    return ''.join(GANAS[x] for x in (recipe or '').split())

# The common samavṛtta metres, written as gaṇa *recipes* rather than as weight patterns: a recipe
# is what a handbook prints and what can be checked by eye, and the pattern is derived from it. A
# hand-copied 21-syllable bit pattern is a silent bug waiting to happen.
METERS = {
    'śālinī':           'ma ta ta ga ga',
    'indravajrā':       'ta ta ja ga ga',
    'upendravajrā':     'ja ta ja ga ga',
    'rathoddhatā':      'ra na ra la ga',
    'svāgatā':          'ra na bha ga ga',
    'vaṃśastha':        'ja ta ja ra',
    'indravaṃśā':       'ta ta ja ra',
    'drutavilambita':   'na bha bha ra',
    'toṭaka':           'sa sa sa sa',
    'bhujaṅgaprayāta':  'ya ya ya ya',
    'praharṣiṇī':       'ma na ja ra ga',
    'rucirā':           'ja bha sa ja ga',
    'vasantatilakā':    'ta bha ja ja ga ga',
    'mālinī':           'na na ma ya ya',
    'pṛthvī':           'ja sa ja sa ya la ga',
    'mandākrāntā':      'ma bha na ta ta ga ga',
    'śikhariṇī':        'ya ma na sa bha la ga',
    'hariṇī':           'na sa ma ra sa la ga',
    'śārdūlavikrīḍita': 'ma sa ja sa ta ta ga',
    'sragdharā':        'ma ra bha na ya ya ya',
}
_PAT = {k: gana_pattern(v) for k, v in METERS.items()}
# upajāti is not a metre but a licence: any mix of these two across the four pādas of one verse.
_MIXED = {'upajāti': ('indravajrā', 'upendravajrā')}

def _pada_ok(w, pat) -> bool:
    'Does one pāda of weights fit a pattern? The last syllable of a pāda is anceps, always.'
    return len(w) == len(pat) and all(x == (c == 'g') for x, c in zip(w[:-1], pat[:-1]))

# The śloka is a syllable *count* plus a cadence, not a fixed pattern. The even pādas carry the
# strict rule — 5 laghu, 6 guru, 7 laghu — while the odd ones may take `pathyā` or one of four
# licensed `vipulā` shapes. Checking that cadence is what separates a śloka from any other 32
# syllables: without it, thirty-two repetitions of `ka` are an anuṣṭubh, and so is most prose of
# the right length.
_PATHYA = 'lgg'
_VIPULA = {'lll':'na-vipulā', 'gll':'bha-vipulā', 'ggg':'ma-vipulā', 'glg':'ra-vipulā'}

def _anustubh(qs):
    'The śloka variant when four 8-syllable pādas scan as one, else None.'
    if not all(list(q[4:7]) == [False, True, False] for q in qs[1::2]): return None
    var = set()
    for q in qs[0::2]:
        k = ''.join('g' if x else 'l' for x in q[4:7])
        if k == _PATHYA: var.add('pathyā')
        elif k in _VIPULA: var.add(_VIPULA[k])
        else: return None
    return ' '.join(sorted(var))

# What is printed *with* a verse but is not part of it. The citation is the trap and it is
# unmissable once seen: `|| Manu_1.1 ||` contributes `ma`+`nu` to the scan, so every verse in a
# GRETIL file comes out 34 syllables instead of 32 and matches no metre whatsoever. A `>` gloss is
# a translation — often English — and a heading is not spoken at all.
_BARENUM = re.compile(r'(?:\|\||//|॥)\s*[\d०-९]+\s*(?:\|\||//|॥)')
_NOSCAN  = re.compile(r'^[ \t]*>[^\n]*'          # a gloss or translation line
                      r'|^[ \t]*#{1,6}[^\n]*'    # a heading
                      r'|\[[^\]\n]*\]', re.M)    # [h: ... :h], [page 3], the [57M] variant number

def metrical_text(s:str) -> str:
    """A verse with everything removed that is printed beside it but does not scan.

    `CITE_RE` does the citation rather than a looser pattern on purpose: it insists on a siglum and
    a number between the daṇḍa pairs, so `// Mn_1.1 //` goes and the `/ pāda /` separators of a
    GRETIL romanised verse stay."""
    return _NOSCAN.sub(' ', _BARENUM.sub(' ', CITE_RE.sub(' ', s or '')))

# --- mātrā metres: the āryā family --------------------------------------------------------------
# A second, older way to build a verse. `METERS` above counts *syllables* in a fixed order of
# weights; the āryā family counts **morae** — a laghu is one mātrā, a guru is two — and fills a
# fixed number of them with whatever syllables the poet likes. So a syllable count says nothing
# here: the four verses of the Sāṃkhyakārikā tested below run 32, 35, 41 and 37 syllables and are
# all the same metre.
#
# The unit is the *caturmātra* gaṇa, four morae, of which there are exactly five shapes. A half is
# a fixed ladder of them:
#
# | half | structure | morae |
# |---|---|---|
# | pūrvārdha  | 7 caturmātra gaṇas + one guru          | 30 |
# | uttarārdha | as above, but the 6th gaṇa is a single laghu | 27 |
# | āryāgīti   | 8 caturmātra gaṇas                     | 32 |
#
# and the five metres of the family are the ways of pairing two halves. Two constraints do the
# real discriminating work, and without them a mora count alone would match almost anything: the
# **odd** gaṇas (1st, 3rd, 5th, 7th) may not be the ja-gaṇa `˘ ¯ ˘`, and the **6th** gaṇa must be
# either that same ja-gaṇa or four laghus. Both are visible in the Sāṃkhyakārikā, where `lgl` turns
# up constantly in even positions and never once in an odd one.
_G, _S = 'gana', 'syl'
_PURVA  = [(_G,4)]*7 + [(_S,0)]                     # 30 morae
_UTTARA = [(_G,4)]*5 + [(_G,1), (_G,4), (_S,0)]     # 27 morae — the 6th gaṇa shrinks to one laghu
_GITI8  = [(_G,4)]*8                                # 32 morae

MATRA_METERS = {
    'āryā':     (_PURVA,  _UTTARA),   # 30 + 27
    'gīti':     (_PURVA,  _PURVA),    # 30 + 30
    'upagīti':  (_UTTARA, _UTTARA),   # 27 + 27
    'udgīti':   (_UTTARA, _PURVA),    # 27 + 30 — the āryā halves the other way round
    'āryāgīti': (_GITI8,  _GITI8),    # 32 + 32
}
CATURMATRA = ('gg', 'gll', 'llg', 'lgl', 'llll')    # the five four-mora gaṇas; `lgl` is the ja-gaṇa

def matras(text:str) -> int:
    'The morae of a string: one for a laghu, two for a guru.'
    return sum(2 if h else 1 for _, h in syllables(text))

def _fit_half(w, slots):
    """Fill one half's ladder of gaṇas from a list of weights; the gaṇa patterns, or None.

    Every boundary has to fall *between* syllables — a gaṇa that would need half of a guru is not a
    gaṇa — which is what makes this a real constraint rather than arithmetic on a total. `w` arrives
    with its last syllable already read as guru, since the syllable closing a half is anceps."""
    i, pats = 0, []
    for kind, want in slots:
        if kind == _S:                              # the closing guru: one syllable, whatever it is
            if i >= len(w): return None
            pats.append('g'); i += 1; continue
        s, j, pat = 0, i, ''
        while j < len(w) and s < want:
            s += 2 if w[j] else 1
            pat += 'g' if w[j] else 'l'
            j += 1
        if s != want: return None                   # overshot: the boundary is mid-syllable
        pats.append(pat); i = j
    return pats if i == len(w) else None

def _half_ok(pats, slots) -> bool:
    'The two gaṇa constraints that separate an āryā from any other 57 morae.'
    if any(pats[i] == 'lgl' for i in (0, 2, 4, 6) if i < len(pats)): return False
    six = pats[5] if len(pats) > 5 else ''
    return six == 'l' if slots[5][1] == 1 else six in ('lgl', 'llll')

def detect_matra_meter(text:str, weights:list=None):
    """The āryā-family metre of a verse — `AttrDict(name, halves, ganas, matras)` — or None.

    The split between the halves is *searched* rather than read off the punctuation, because the
    daṇḍa between them is not reliably present and a half boundary is anceps: the syllable closing
    each half counts as guru however it is written. There are only as many candidate splits as
    syllables, and the ladder pins everything else, so this stays cheap."""
    w = weights if weights is not None else [h for _, h in syllables(metrical_text(text))]
    for k in range(4, len(w)):
        a, b = list(w[:k]), list(w[k:])
        if not b: continue
        a[-1] = b[-1] = True                        # each half closes on an anceps, read as guru
        for nm, (s1, s2) in MATRA_METERS.items():
            p1, p2 = _fit_half(a, s1), _fit_half(b, s2)
            if p1 and p2 and _half_ok(p1, s1) and _half_ok(p2, s2):
                m1 = sum(2 if h else 1 for h in a)
                m2 = sum(2 if h else 1 for h in b)
                return AttrDict(name=nm, halves=(m1, m2), matras=m1 + m2,
                                ganas=' '.join(p1) + ' | ' + ' '.join(p2))
    return None

def detect_meter(text:str):
    """The metre of one verse — `AttrDict(name, variant, syllables, per_pada, ganas, scan, matras)`.

    Both systems are tried, syllable-counting first. A **varṇa** metre (`METERS`) fixes the weight
    of every syllable in a pāda; a **mātrā** metre (`MATRA_METERS`, the āryā family) fixes only the
    morae, so its verses vary in syllable count and `per_pada` is meaningless for them — the
    Sāṃkhyakārikā runs 32, 35, 41 and 37 syllables in four consecutive āryās.

    `name` is `None` when the verse divides into four equal pādas but fits neither catalogue. That
    is a statement about the catalogue, **not** a claim that the text is or is not verse: nothing
    here is a verse/prose classifier, which is why the anuṣṭubh branch insists on the cadence
    instead of accepting any 32 syllables.

    `ganas` reports the signature of the **first pāda** with its final syllable read as guru, since
    that syllable is anceps and two verses in one metre would otherwise fail to compare equal. For a
    mātrā metre it reports the caturmātra decomposition of both halves instead, which is what there
    is to compare there — the ladder is the same in every āryā, the filling is the poet's.

    The text is passed through `metrical_text` first, so a verse may be handed over exactly as it is
    stored — citation, gloss and all — and still scan as the verse it is."""
    w = [h for _, h in syllables(metrical_text(text))]
    n = len(w)
    if n < 8: return None
    sc, mt = ''.join('g' if h else 'l' for h in w), sum(2 if h else 1 for h in w)
    if not n % 4:
        q  = n // 4
        qs = [w[i*q:(i+1)*q] for i in range(4)]
        sig = ' '.join(ganas(sc[:q][:-1] + 'g'))
        mk = lambda nm, var=None: AttrDict(name=nm, variant=var, syllables=n, per_pada=q,
                                           ganas=sig, scan=sc, matras=mt, halves=None)
        for nm, pat in _PAT.items():
            if len(pat) == q and all(_pada_ok(x, pat) for x in qs): return mk(nm)
        for nm, (a, b) in _MIXED.items():
            pa, pb = _PAT[a], _PAT[b]
            if len(pa) == q and all(_pada_ok(x, pa) or _pada_ok(x, pb) for x in qs): return mk(nm)
        if q == 8 and (v := _anustubh(qs)) is not None: return mk('anuṣṭubh', v)
    # only now the mora-counting family, so a varṇa metre is never relabelled by it
    if (a := detect_matra_meter(text, w)):
        return AttrDict(name=a.name, variant=None, syllables=n, per_pada=None,
                        ganas=a.ganas, scan=sc, matras=a.matras, halves=a.halves)
    if n % 4: return None
    return AttrDict(name=None, variant=None, syllables=n, per_pada=n//4,
                    ganas=' '.join(ganas(sc[:n//4][:-1] + 'g')), scan=sc, matras=mt, halves=None)

# --- metre as chunk metadata --------------------------------------------------------------------
def verse_units(text:str) -> L:
    'The individual verses inside one chunk — a chunk may pack several, and metre is per verse.'
    return L(t for _, _, t, _ in verse_spans(text)) or L([(text or '').strip()])

def verse_meta(text:str) -> dict:
    """The metrical facets of one chunk, as the dict that becomes its `metadata` JSON.

    `get_store` already indexes `metadata` for FTS beside `content`, so writing metre *here* rather
    than into a column of its own is what makes `db.search('mandākrāntā')` and a `where` on the same
    field work with no schema change and no cooperation from any caller — including
    [vishalakshi](https://github.com/vedicreader/vishalakshi), which threads `where` straight down
    to `doc_search`.

    The gaṇa signature is joined with `_` on purpose. The tokenizer chain treats `_` as a word
    joiner (it is why `fts_search` survives as one token), so `ma_bha_na_ta_ta_ga_ga` stays a single
    searchable term instead of becoming seven meaningless ones. That is what makes *find every verse
    shaped like this one* a query rather than a scan.

    Keys are omitted rather than written empty, so a chunk of prose costs `{}` and nothing else."""
    ms = L(verse_units(text)).map(detect_meter).filter(lambda m: m is not None)
    if not ms: return {}
    out, named = {}, ms.filter(lambda m: m.name)
    if named:
        out['meter'] = ' '.join(dict.fromkeys(named.attrgot('name')))
        if (vs := [v for v in dict.fromkeys(named.attrgot('variant')) if v]): out['variant'] = ' '.join(vs)
    # A varṇa metre's signature is shared by every verse in it, so it is a join key worth an index.
    # A mātrā metre's gaṇa filling is the poet's choice and is close to unique per verse, so what
    # goes in is the mora shape (`30+27`) — the facet two texts can actually be compared on.
    var, mat = ms.filter(lambda m: m.per_pada), ms.filter(lambda m: m.halves)
    if (gs := dict.fromkeys(m.ganas.replace(' ', '_') for m in var if m.ganas)): out['gana'] = ' '.join(gs)
    if (ps := dict.fromkeys(str(m.per_pada) for m in var)): out['pada'] = ' '.join(ps)
    if (hs := dict.fromkeys('+'.join(map(str, m.halves)) for m in mat)): out['matra'] = ' '.join(hs)
    return out

# --- lemmas: the one thing the deterministic fold cannot do -------------------------------------
def lemma_facets(text:str,          # chunk text
                 nlp,               # a `stanza_pipe('sa')` pipeline (or any spaCy-shaped one)
                 max_terms:int=96   # cap, so one chunk's metadata cannot outgrow its content
                 ) -> dict:
    """`{'lemma': 'gam vac ...'}` for a chunk — the dictionary forms behind its inflected words.

    **Why this is not in the tokenizer.** The `sanskrit` FTS5 tokenizer runs inside SQLite's index
    path, on every token of every insert *and every query*, and a store's tokenizer is fixed when
    its table is created — so whatever goes in there has to be present on every connection that
    ever opens the database. A neural pipeline fails all three tests at once: it is orders of
    magnitude too slow for the query path, it would make the file unreadable without torch and a
    downloaded model, and its output changes with the model version, so an index built today would
    not match a query tomorrow. The fold stays deterministic and table-driven; lemmatisation is
    computed **once at ingest** and written to `metadata`, which `get_store` already indexes for
    FTS beside `content`. Zero read-path cost, no schema change, no caller changes.

    **Only lemmas that differ from the surface are kept.** The surface form is already in `content`
    and already indexed; storing it twice buys nothing and doubles the column. What is worth
    storing is exactly the difference — `gacchati` in the text, `gam` in the index — which is the
    case where a reader who types the dictionary form currently gets nothing.

    **What this does not solve.** Stanza does not split sandhi and does not decompose compounds:
    the Vedic treebank it is trained on is distributed pre-segmented, so `dharmakṣetre` comes back
    as one token. That is the harder half of Sanskrit retrieval and it stays open — this handles
    inflection, which is the half a lemmatiser can honestly claim."""
    if not (nlp and (text or '').strip()): return {}
    try: doc = nlp(metrical_text(text))
    except Exception: return {}
    out = {}
    for t in doc:
        lem, surf = (getattr(t, 'lemma_', '') or '').strip(), (t.text or '').strip()
        if not lem or getattr(t, 'is_punct', False) or getattr(t, 'is_digit', False): continue
        if fold_token(lem) == fold_token(surf): continue     # nothing the surface index lacks
        if len(lem) < 2: continue
        out[lem] = None
        if len(out) >= max_terms: break
    return {'lemma': ' '.join(out)} if out else {}

def sanskrit_meta(nlp=None):
    """The `Profile.meta` callable: metre always, lemmas when a pipeline is supplied.

    A factory rather than a flag because `Profile.meta` is called as `meta(chunk_text)` — the
    pipeline has to be closed over, and closing over it here is also what keeps the model out of
    the picture entirely for the default profiles."""
    if nlp is None: return verse_meta
    def meta(text:str) -> dict: return {**verse_meta(text), **lemma_facets(text, nlp)}
    return meta

@patch
def by_lemma(self:Database,
             lemma:str,           # a dictionary form, e.g. `gam`
             store:str='store',
             prefix:str=None,
             columns:list=None,
             limit:int=50) -> list:
    'Chunks whose verses inflect `lemma`. Needs an index built with `register_profiles(nlp=...)`.'
    cols = list(dict.fromkeys((columns or ['content']) + ['metadata', 'node_id', 'doc_id', 'page']))
    return self.t[ifnone(prefix, '') + store](
        select=', '.join(cols), where='metadata like :sa_l',
        where_args={'sa_l': f'%"lemma": "%{lemma}%'}, limit=limit)

@patch
def by_meter(self:Database,
             meter:str=None,      # metre name, e.g. `mandākrāntā`
             gana:str=None,       # gaṇa signature, `ma bha na ta ta ga ga` or `ma_bha_na_...`
             store:str='store',
             prefix:str=None,
             columns:list=None,
             limit:int=50) -> list:
    """Chunks whose verses are in a metre, or share a gaṇa signature. A filter, not a ranking.

    Comparing how two texts use a metre is a question about *all* the verses in it, so this is a
    `LIKE` over the facets `verse_meta` wrote rather than a search that returns the best few. Pair
    it with a `kind` or `doc_id` filter to put two works side by side."""
    wh, wa = [], {}
    if meter: wh.append('metadata like :sa_m'); wa['sa_m'] = f'%"meter": "%{meter}%'
    if gana:  wh.append('metadata like :sa_g'); wa['sa_g'] = f'%{gana.strip().replace(" ", "_")}%'
    if not wh: return []
    cols = list(dict.fromkeys((columns or ['content']) + ['metadata', 'node_id', 'doc_id', 'page']))
    return self.t[ifnone(prefix, '') + store](select=', '.join(cols), where=' AND '.join(wh),
                                              where_args=wa, limit=limit)

In [ ]:
#| hide
# The gaṇa table checked against the mnemonic it comes from, rather than trusted: read three
# syllables of `yamātārājabhānasalagam` from each starting point and you get that gaṇa's pattern.
_MNE = 'ya mā tā rā ja bhā na sa la gam'
_MW  = ''.join('g' if h else 'l' for _, h in syllables(_MNE))
assert _MW == 'lgggl' + 'glllg', _MW              # ya mā tā rā ja bhā na sa la gam
for _i, _g in enumerate(['ya','ma','ta','ra','ja','bha','na','sa']):
    assert GANAS[_g] == _MW[_i:_i+3], (_g, GANAS[_g], _MW[_i:_i+3])
assert GANAS['ga'] == 'g' and GANAS['la'] == 'l'
assert len({v for v in GANAS.values() if len(v) == 3}) == 8      # the eight are distinct
# recipes round-trip through their patterns
for _n, _r in METERS.items(): assert ' '.join(ganas(gana_pattern(_r))) == _r, _n

# --- weights ------------------------------------------------------------------------------------
# an aspirate is ONE consonant: `atha` is light-light, `artha` is heavy-light
assert scan('atha') == 'll' and scan('artha') == 'gl'
assert scan('rāma') == 'gl'                       # long vowel is guru
assert scan('kaṃsa') == 'gl'                      # anusvāra closes the syllable
assert scan('duḥkha') == 'gl'                     # so does visarga
assert scan('mitra') == 'gl'                      # two consonants after the vowel
# a consonant cluster counts ACROSS a word boundary — `tat sarvam` closes on `t`+`s`
assert scan('tat sarvam') == 'ggg', scan('tat sarvam')

# --- word boundaries must not manufacture diphthongs --------------------------------------------
# `jīva iti` is four syllables. Scanning the whitespace-stripped string fuses `a`+`i` into `ai`
# and gives three — one short pāda, and the verse then matches no metre at all.
assert [s for s, _ in syllables('jīva iti')] == ['jī','va','i','ti'], syllables('jīva iti')
assert len(syllables('naiva')) == 2               # a real diphthong inside a word still is one
assert len(syllables('ca api')) == 3
assert len(syllables('agni-kuṇḍa')) == 4          # a hyphen fences too: no `i`+`k` fusion of vowels

# --- metres -------------------------------------------------------------------------------------
_VERSES = [
 ('anuṣṭubh', 'dharmakṣetre kurukṣetre samavetā yuyutsavaḥ | '
              'māmakāḥ pāṇḍavāś caiva kim akurvata sañjaya ||'),
 ('anuṣṭubh', 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः । मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥'),
 ('mandākrāntā', 'kaścit kāntāvirahaguruṇā svādhikārāt pramattaḥ '
                 'śāpenāstaṃgamitamahimā varṣabhogyeṇa bhartuḥ | '
                 'yakṣaś cakre janakatanayāsnānapuṇyodakeṣu '
                 'snigdhacchāyātaruṣu vasatiṃ rāmagiryāśrameṣu ||'),
 ('vasantatilakā', 'vidyud vibhāti jaladeṣu yathā tathaiva '
                   'dehe sya jīva iti bhāti sadā vibhinnaḥ | '
                   'yasmād idaṃ jagad aśeṣam abhūt purastāt '
                   'taṃ vai prapadya śaraṇaṃ paramaṃ prapadye ||'),
]
for _want, _v in _VERSES:
    _m = detect_meter(_v)
    assert _m and _m.name == _want, (_want, _m and _m.name, _m and _m.scan)
# the same verse in either script scans identically — length survives deva2iast
assert detect_meter(_VERSES[0][1]).scan == detect_meter(_VERSES[1][1]).scan
# and the signature is the metre's own recipe
assert detect_meter(_VERSES[2][1]).ganas == METERS['mandākrāntā']

# --- a syllable count is NOT a metre -------------------------------------------------------------
# 32 syllables of nothing must not become an anuṣṭubh: the even-pāda cadence is what decides.
for _junk in ['ka ' * 32, 'a ' * 32, 'rā ' * 32]:
    assert detect_meter(_junk).name is None, (_junk[:8], detect_meter(_junk).name)
# ordinary unmetred prose of a length that divides by four, likewise
_prose = ('yathā hi puruṣaḥ svapne nānāvidhāni rūpāṇi paśyati tathaiva ayaṃ jīvaḥ saṃsāre bhramati '
          'tasmāt sa vidvān na śocati')
assert (detect_meter(_prose) or AttrDict(name=None)).name is None
assert detect_meter('rāmaḥ') is None              # too short to be anything
assert detect_meter('') is None

# --- variants ------------------------------------------------------------------------------------
assert detect_meter(_VERSES[0][1]).variant in ('pathyā', 'pathyā'), detect_meter(_VERSES[0][1]).variant

# --- the metadata a chunk carries ----------------------------------------------------------------
_md = verse_meta(_VERSES[2][1])
assert _md['meter'] == 'mandākrāntā' and _md['pada'] == '17'
assert _md['gana'] == 'ma_bha_na_ta_ta_ga_ga', _md          # one token, joined for the tokenizer
assert verse_meta('The quick brown fox.') == {} or 'meter' not in verse_meta('The quick brown fox.')
# a chunk holding two verses in different metres reports both
_two = _VERSES[2][1] + '\n' + _VERSES[3][1]
assert set(verse_meta(_two)['meter'].split()) == {'mandākrāntā', 'vasantatilakā'}, verse_meta(_two)

# --- mātrā metres: the āryā family ---------------------------------------------------------------
# The Sāṃkhyakārikā is composed throughout in āryā. Four consecutive verses of it run 32, 35, 41
# and 37 syllables — which is the whole point of a mora-counting metre, and the reason a syllable
# count identifies nothing here.
_ARYAS = [
 "duḥkhatrayābhighātāj jijñāsā tadapaghātake hetau | "
 "dṛṣṭe sāpārthā cen naikāntātyantato 'bhāvāt ||",
 "dṛṣṭavad ānuśravikaḥ sa hy aviśuddhikṣayātiśayayuktaḥ | "
 "tadviparītaḥ śreyān vyaktāvyaktajñavijñānāt ||",
 "mūlaprakṛtir avikṛtir mahadādyāḥ prakṛtivikṛtayaḥ sapta | "
 "ṣoḍaśakas tu vikāro na prakṛtir na vikṛtiḥ puruṣaḥ ||",
 "dṛṣṭam anumānam āptavacanaṃ ca sarvapramāṇasiddhatvāt | "
 "trividhaṃ pramāṇam iṣṭaṃ prameyasiddhiḥ pramāṇād dhi ||",
]
for _a in _ARYAS:
    _m = detect_meter(_a)
    assert _m and _m.name == 'āryā', (_m and _m.name, _m and _m.scan)
    assert _m.halves == (30, 27), _m.halves
    assert _m.per_pada is None                    # a mātrā metre has no fixed pāda length
assert len({detect_meter(_a).syllables for _a in _ARYAS}) == 4, 'the point is that they differ'
assert matras('rāma') == 3 and matras('ka') == 1  # guru is two morae, laghu one

# the ja-gaṇa rule is real and it is what the odd/even distinction rests on: `lgl` turns up all
# over the even positions of these verses and never once in an odd one
for _a in _ARYAS:
    _p = detect_meter(_a).ganas.split(' | ')[0].split()
    assert all(_p[i] != 'lgl' for i in (0, 2, 4, 6)), _p
    assert _p[5] in ('lgl', 'llll'), _p           # the 6th gaṇa is ja or four laghus

# a varṇa metre must never be relabelled by the mora pass, which runs second for exactly this reason
for _want, _v in _VERSES: assert detect_meter(_v).name == _want
assert detect_meter(_VERSES[0][1]).halves is None

# --- the constraints have to do the discriminating ------------------------------------------------
# 57 morae is not an āryā. Random weight strings of *exactly* the right mora count must still be
# rejected by the gaṇa ladder, or the metre would mean nothing.
import random as _rnd
_r = _rnd.Random(0)
def _rand_syl(target):
    _s, _out = 0, []
    while _s < target:
        _h = _r.random() < .5
        if _s + (2 if _h else 1) > target: _h = False
        _out.append('rā' if _h else 'ka'); _s += 2 if _h else 1
    return ' '.join(_out)
_fp = sum(1 for _ in range(400) if (detect_matra_meter(_rand_syl(57)) is not None))
assert _fp == 0, f'{_fp}/400 random 57-mora strings matched an āryā-family metre'
# and real prose does not
for _p in ['atha khalu bhagavān etad avocat yathā hi puruṣaḥ svapne nānāvidhāni rūpāṇi paśyati '
           'tathaiva ayaṃ jīvaḥ saṃsāre bhramati tasmād vidvān na śocati',
           'ka ' * 32, 'The quick brown fox jumps over the lazy dog and runs into the woods']:
    assert detect_matra_meter(_p) is None, _p[:30]

# --- the whole family round-trips ------------------------------------------------------------------
# Built from each ladder by construction, every metre must come back as itself. `udgīti` is the
# āryā halves the other way round, so it is the one that catches a swapped pair.
_CAT = {'gg':[True,True], 'gll':[True,False,False], 'llg':[False,False,True],
        'lgl':[False,True,False], 'llll':[False]*4}
def _build(slots, rng):
    _w = []
    for _i, (_k, _want) in enumerate(slots):
        if _k == 'syl': _w.append(True); continue
        if _want == 1:  _w.append(False); continue
        if   _i == 5: _o = ['lgl', 'llll']              # the 6th gaṇa rule
        elif _i == 7: _o = ['gg', 'llg']                # an āryāgīti's 8th gaṇa ends guru
        elif _i in (0, 2, 4, 6): _o = [k for k in _CAT if k != 'lgl']
        else: _o = sorted(_CAT)
        _w += _CAT[rng.choice(sorted(_o))]
    return _w
_rg = _rnd.Random(7)
for _nm, (_s1, _s2) in MATRA_METERS.items():
    for _ in range(50):
        _got = detect_matra_meter('', _build(_s1, _rg) + _build(_s2, _rg))
        assert _got and _got.name == _nm, (_nm, _got and _got.name)

# --- and the metadata a mātrā verse carries ---------------------------------------------------------
_am = verse_meta(_ARYAS[0])
assert _am['meter'] == 'āryā' and _am['matra'] == '30+27', _am
assert 'pada' not in _am and 'gana' not in _am, 'a per-verse gaṇa filling is not a join key'


In [ ]:
#| hide
# A phone spent as one syllable's coda is not also the next syllable's onset. Weights were always
# right; the readable segmentation was not, and `syllables` is exported.
assert [s for s, _ in syllables('kaṃsa')]  == ['kaṃ', 'sa']
assert [s for s, _ in syllables('duḥkha')] == ['duḥ', 'kha']
assert scan('kaṃsa') == 'gl' and scan('duḥkha') == 'gl'          # unchanged
assert ''.join(s for s, _ in syllables('kaṃsa')) == 'kaṃsa'      # no phone duplicated or dropped


## Readers

Four formats, one shape: `(pages, meta)`, where a page is one structural unit. What each reader
mostly does is *decline* to index things — GRETIL's 45-line provenance header, repeated across
5,000 files and the single biggest source of false matches in that corpus; TEI's `<app>/<rdg>`
variant readings, which would file several spellings of one line as if they were different verses;
vedicreader's `exclude_from_display` audio cues.

In [ ]:
#| export
_GRETIL_END = re.compile(r'COPYRIGHT AND TERMS OF USAGE AS FOR SOURCE FILE\.|'
                         r'For further information see:\s*\n\s*http\S+', re.I)

def _detag(t:str) -> str:
    import html
    t = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', t, flags=re.S | re.I)
    return html.unescape(re.sub(r'<[^>]+>', '', t))

def gretil_parse(src) -> tuple:
    """GRETIL plain-text (`*_u.htm`) to `(pages, meta)`.

    The ~45-line GRETIL header — provenance, the editions collated, and a full Unicode diacritics
    table — is boilerplate repeated across every one of the 5,000 files. Indexed, it is the single
    biggest source of false matches in the corpus, so it is cut at the licence banner and kept in
    `meta` rather than thrown away."""
    txt = _detag(Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src))
    head, body = '', txt
    if (m := _GRETIL_END.search(txt)):
        head, body = txt[:m.end()], txt[m.end():]
        # the diacritics table sits *after* the banner; drop lines up to the last of its entries
        if (d := list(re.finditer(r'^\s*\w[\w\s]*\s{2,}[^\x00-\x7f]\s*$', body, re.M))):
            head, body = head + body[:d[-1].end()], body[d[-1].end():]
    body = re.sub(r'\n{3,}', '\n\n', body).strip()
    # Editorial notes survive past the licence banner ("accents have been dropped in order to
    # facilitate word search", mirror URLs). They are pure ASCII with no daṇḍa and no citation, so
    # leading blocks that carry no Sanskrit signal at all are moved into `meta`, not indexed.
    blocks = [b for b in re.split(r'\n\s*\n', body) if b.strip()]
    while blocks and not _has_sanskrit(blocks[0]):
        head += '\n\n' + blocks.pop(0)
    body = '\n\n'.join(blocks)
    # One page per *verse*, not per blank-line block. GRETIL runs verses on consecutive lines with
    # no blank line between them, so a whole adhyāya arrives as one block; `build_tree` reads the
    # first citation of a page to place it, and would file every verse of Manu's chapter 5 under
    # chapter 4 — the chapter node simply never appears.
    spans = verse_spans(body)
    pages = [(i, s) for i, (_, _, s, _) in enumerate(spans)] if len(spans) > 1 else list(enumerate(blocks))
    ttl = _first_line(head) or (Path(src).stem if _isfile(src) else 'gretil')
    return pages or [(0, body)], dict(fmt='gretil', title=ttl, header=head.strip()[:2000])

def tei_parse(src) -> tuple:
    """GRETIL `corpustei` / SARIT TEI to `(pages, meta)`.

    `<lg xml:id="Manu_1.1">` is the verse and carries its own citation, `<div type="adhyāya">` is
    the section, and `<seg type="pāda">` is below the unit anyone retrieves. Two things are
    deliberately *not* indexed as primary text: `<app>/<rdg>` variant readings, which would put
    several spellings of the same line into the store as if they were different verses, and
    `<note type="analysis">`, GRETIL's sandhi-split duplicate of the verse it follows."""
    from xml.etree import ElementTree as ET
    raw = Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src)
    root = ET.fromstring(raw)
    ns = {'t': 'http://www.tei-c.org/ns/1.0'}
    def tag(e): return e.tag.split('}')[-1]
    def txt(e):
        parts = []
        for n in e.iter():
            if tag(n) in ('note', 'app', 'rdg', 'lem'): continue
            if n.text: parts.append(n.text)
            if n is not e and n.tail: parts.append(n.tail)
        return re.sub(r'\s+', ' ', ' '.join(parts)).strip()
    body = root.find('.//t:text/t:body', ns) or root.find('.//text/body') or root
    pages, path = [], []
    def walk(el, depth=0):
        for ch in el:
            t = tag(ch)
            if t == 'div':
                nm = ch.get('type') or 'div'; n = ch.get('n') or ''
                path.append(f"{nm} {n}".strip())
                pages.append((len(pages), f"{'#' * min(depth + 2, 6)} {path[-1]}"))
                walk(ch, depth + 1); path.pop()
            elif t == 'lg':
                cid = ch.get('{http://www.w3.org/XML/1998/namespace}id') or ch.get('id') or ''
                # `<l>` often already carries a trailing `//`; appending the id beside it would
                # leave `// || Manu_1.2 ||` and two boundary markers where there is one boundary.
                s = re.sub(r'\s*(?://|\|\|)\s*$', '', txt(ch))
                if s: pages.append((len(pages), f"{s} || {cid} ||" if cid else s))
            elif t in ('p', 'ab'):
                if (s := txt(ch)): pages.append((len(pages), s))
            elif t in ('note', 'app'): continue
            else: walk(ch, depth)
    walk(body)
    ttl = (root.findtext('.//t:titleStmt/t:title', default='', namespaces=ns)
           or root.findtext('.//titleStmt/title', default='') or '').strip()
    return _merge_pages(pages), dict(fmt='tei', title=ttl or (Path(src).stem if _isfile(src) else 'tei'))

def vr_xml_parse(src) -> tuple:
    """vedicreader `<lyrics>` XML to `(pages, meta)`.

    `<section>` is already the retrieval unit and `<line>` the pāda, so this parser mostly gets out
    of the way. The one judgement call: `meaning`/`etymology` glosses are emitted beside the verse
    rather than dropped, because a corpus where the Sanskrit and its gloss are searched together is
    the reason to have the gloss at all — and `ignore`/`exclude_from_display` lines (filler music
    cues, playback markers) are dropped, since they are audio-sync artefacts, not text."""
    from xml.etree import ElementTree as ET
    raw = Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src)
    root = ET.fromstring(raw)
    pages, ttl = [], ''
    for so, s in enumerate(root.iter('section')):
        nm = s.get('name') or f'section {so}'
        lines = [l for l in s.iterfind('line')
                 if (l.text or '').strip() and (l.get('exclude_from_display') or 'false').lower() != 'true']
        if nm == 'title' and lines and not ttl: ttl = (lines[0].text or '').strip()
        if not lines: continue
        pages.append((len(pages), f'## {nm}'))
        # Each gloss goes directly under the pāda it glosses rather than being concatenated at the
        # end of the section. Pooled, a long section's glosses form a single block bigger than the
        # verse itself, and the split that then has to happen cuts the translation loose from the
        # Sanskrit — a chunk of pure English that matches any paraphrase of it.
        buf = []
        for l in lines:
            buf.append((l.text or '').strip())
            if (g := (l.get('meaning') or l.get('caption') or '').strip()): buf.append('> ' + g)
        body = '\n'.join(buf)
        if not re.search(r'[।॥]\s*$', body.split('\n>')[0]): body += '\n॥'
        if (sm := (s.get('meaning') or '').strip()): body += '\n> ' + sm
        pages.append((len(pages), body))
    cat = (root.findtext('category') or '').strip()
    tags = (root.findtext('tags') or '').strip()
    return _merge_pages(pages), dict(fmt='vedicreader', title=ttl, category=cat, tags=tags)

def dcs_parse(src) -> tuple:
    """DCS / ambuda analysed text to `(pages, meta)`.

    Each verse is a `# id = R.1.1.1` block of `surface⇥lemma⇥features` lines. The surface forms
    rebuild the readable verse; the lemmas are kept beside it because they are a sandhi-split
    index of exactly the words a compound hides — the one thing FTS on raw Sanskrit cannot see."""
    raw = Path(src).read_text(encoding='utf-8', errors='replace') if _isfile(src) else str(src)
    pages, cur, lem, cid = [], [], [], None
    def flush():
        if not cur: return
        v = ' '.join(cur)
        # lemmas on the verse's own page, so they travel with it rather than opening the next unit
        pages.append((len(pages), (f"{v} || {cid} ||" if cid else v)
                      + ('\n> lemmas: ' + ' '.join(dict.fromkeys(lem)) if lem else '')))
    for ln in raw.splitlines():
        if ln.startswith('#'):
            if (m := re.search(r'id\s*=\s*(\S+)', ln)): flush(); cur, lem, cid = [], [], m.group(1)
            continue
        if not ln.strip():
            flush(); cur, lem, cid = [], [], None
            continue
        f = ln.split('\t')
        if f and f[0].strip(): cur.append(f[0].strip())
        if len(f) > 1 and f[1].strip(): lem.append(f[1].strip())
    flush()
    return _merge_pages(pages), dict(fmt='dcs', title=(Path(src).stem if _isfile(src) else 'dcs'))

def _isfile(x):
    try: return Path(x).is_file()
    except (OSError, ValueError): return False

def _first_line(t):
    for l in (t or '').splitlines():
        if (s := l.strip()) and not s.startswith('http') and len(s) > 3: return s[:120]
    return ''

def _merge_pages(pages, per:int=1):
    'One page per structural unit is fine for the tree; collapse to text blocks for build_tree.'
    return [(i, t) for i, t in pages if (t or '').strip()]

_IAST_DIAC = re.compile('[āīūṛṝḷḹṅñṭḍṇśṣṃḥĀĪŪṚṄÑṬḌṆŚṢṂḤ]')

def _has_sanskrit(block:str) -> bool:
    'Whether a block carries any Sanskrit signal — script, diacritic, daṇḍa or citation.'
    return bool(_DEVA.search(block) or _IAST_DIAC.search(block)
                or DANDA in block or DDANDA in block or CITE_RE.search(block))

_SANSKRIT_EXTS = '.xml,.tei,.htm,.html,.conllu,.txt'

def is_sanskrit(text:str, thresh:float=0.15) -> bool:
    """Whether `text` reads as Sanskrit — by script, by transliteration, or by structure.

    Latin-script Sanskrit is the hard case, since IAST is ordinary Unicode letters. Two signals
    answer it. Retroflex and long-vowel diacritics (`ṛ ṇ ṭ ḍ ś ṣ ṃ ḥ ā ī ū`) essentially do not
    occur in English, so even a low density of them is decisive; and citation markers and daṇḍas
    give a structural signal for text that has had its diacritics stripped."""
    s = (text or '')[:20000]
    if not s.strip(): return False
    if len(_DEVA.findall(s)) / max(len(s), 1) > thresh: return True
    if len(_IAST_DIAC.findall(s)) / max(len(s), 1) > 0.01: return True
    n = len(CITE_RE.findall(s)) + s.count(DDANDA)
    return n >= 3 and n / max(len(s.splitlines()), 1) > 0.05

def sanskrit_parse(src) -> tuple:
    'Parse any supported Sanskrit source to `(pages, meta)`, picking the reader by shape.'
    p = Path(src) if _isfile(src) else None
    raw = p.read_text(encoding='utf-8', errors='replace')[:4000] if p else str(src)[:4000]
    sfx = p.suffix.lower() if p else ''
    if sfx in ('.conllu',) or re.search(r'^#\s*id\s*=', raw, re.M): return dcs_parse(src)
    if '<lyrics' in raw: return vr_xml_parse(src)
    if 'tei-c.org' in raw or '<TEI' in raw or '<teiHeader' in raw: return tei_parse(src)
    if sfx in ('.htm', '.html') or 'GRETIL' in raw: return gretil_parse(src)
    if sfx == '.xml':
        return vr_xml_parse(src) if '<lyrics' in raw else tei_parse(src)
    return gretil_parse(src)

In [ ]:
#| hide
from fastcore.all import Path as _P
_F = _P('sanskrit')
if _F.exists():
    _pg, _m = vr_xml_parse(_F/'lalita_excerpt.xml')
    assert _m['fmt'] == 'vedicreader' and _m['category'] == 'stotras' and _pg
    assert any('lagunyasam' in t for _, t in _pg)

    _pg, _m = gretil_parse(_F/'isopanisad_excerpt.htm')
    assert _m['fmt'] == 'gretil'
    _txt = '\n'.join(t for _, t in _pg)
    assert 'IsUp_1' in _txt
    # the GRETIL boilerplate is in meta, not in the corpus
    assert 'COPYRIGHT AND TERMS OF USAGE' not in _txt and 'vocalic r' not in _txt
    assert 'GRETIL' in _m['header']

    _pg, _m = tei_parse(_F/'manu_tei_excerpt.xml')
    assert _m['fmt'] == 'tei' and any('adhyāya' in t for _, t in _pg)
    assert any('Manu_1.1' in t for _, t in _pg)

    _pg, _m = dcs_parse(_F/'dcs_excerpt.txt')
    assert _m['fmt'] == 'dcs' and any('R.1.1.1' in t for _, t in _pg)
    assert any('lemmas:' in t for _, t in _pg)          # the sandhi-split index is kept

    # dispatch picks the right reader from content, not just the extension
    for _f, _fmt in [('lalita_excerpt.xml','vedicreader'), ('manu_tei_excerpt.xml','tei'),
                     ('isopanisad_excerpt.htm','gretil'), ('dcs_excerpt.txt','dcs')]:
        assert sanskrit_parse(_F/_f)[1]['fmt'] == _fmt, _f
assert is_sanskrit('rāmaḥ || R_1.1 ||\nsītā || R_1.2 ||\nlakṣmaṇaḥ || R_1.3 ||')
assert not is_sanskrit('The quick brown fox jumps over the lazy dog, repeatedly and at length.')

## Profiles

Two, because verse and prose want different budgets and nothing else about the pipeline differs.
Prose is not auto-detected — a commentary and its root text share every other signal — so it is
asked for by name: `add_file(path, profile='sanskrit_prose')`.

In [ ]:
#| export
def _sniff(text:str) -> bool:
    """Whether a shared extension (`.xml`, `.txt`, `.htm`) holds Sanskrit this module can read.

    Tags are stripped before the test and the window is wide, because a GRETIL `.htm` opens with a
    `<!DOCTYPE>`, a stylesheet and a 45-line provenance header — several kilobytes of ASCII before
    the first syllable of Sanskrit. Sniffing the head of the raw file finds nothing."""
    raw = (text or '')[:60000]
    if '<lyrics' in raw[:4000]: return True
    # TEI states its language, which beats sniffing: a 16 KB edition can be almost entirely header,
    # and TEI keeps the citation in an `xml:id` attribute where no content sniff will find it.
    if ('tei-c.org' in raw or '<teiHeader' in raw) and re.search(r'xml:lang=["\'](sa|pi|pra)\b', raw): return True
    t = _detag(raw)
    if re.search(r'^#\s*id\s*=', t, re.M): return True
    return is_sanskrit(t)

def register_profiles(nlp=None):
    """Register the Sanskrit profiles. Called on import; safe to call again.

    Two of them, because verse and prose want different budgets and nothing else about the pipeline
    differs. Prose is not auto-detected — a commentary and its root text share every other signal,
    so it is selected by name (`add_file(..., profile='sanskrit_prose')`).

    Both carry `meta=sanskrit_meta(nlp)`, so every chunk ingested through either arrives with its
    metre and gaṇa signature already in the column FTS indexes. Prose gets it too: a bhāṣya quotes
    the verse it is glossing, and that quotation still scans.

    Pass `nlp=stanza_pipe('sa')` to add lemmas to that metadata as well — re-registering replaces
    the profiles by name, so this is the whole opt-in:

    ```python
    from litesearch import stanza_pipe, register_profiles
    register_profiles(nlp=stanza_pipe('sa'))    # once, before add_file/add_dir
    ```

    It is opt-in rather than automatic because it costs a torch install, a model download and a
    real fraction of ingest time, and buys nothing at all for a corpus whose readers type the
    inflected forms they are reading."""
    from litesearch.data import Profile, register_profile
    meta = sanskrit_meta(nlp)
    register_profile(Profile(name='sanskrit_verse', exts='.xml,.tei,.htm,.html,.conllu,.txt',
                             parse=sanskrit_parse, chunker=VerseChunker, mode='verse',
                             detect=_sniff, kind='sanskrit', meta=meta))
    register_profile(Profile(name='sanskrit_prose', exts='', parse=sanskrit_parse,
                             chunker=ProseChunker, mode='verse',
                             detect=lambda _t: False, kind='sanskrit', meta=meta))

register_profiles()

In [ ]:
#| hide
from litesearch.data import profile_for as _pf
from fastcore.all import Path as _P
assert _pf(name='sanskrit_verse').mode == 'verse'
assert _pf(name='sanskrit_prose').detect('anything') is False    # by name only
assert _pf(name='sanskrit_verse').meta is verse_meta             # metre rides in on both profiles
assert _pf(name='sanskrit_prose').meta is verse_meta
_F = _P('sanskrit')
if _F.exists():
    for _f in ['lalita_excerpt.xml', 'manu_tei_excerpt.xml', 'isopanisad_excerpt.htm']:
        assert _pf(_F/_f).name == 'sanskrit_verse', _f
    # a markdown file must not be claimed
    assert _pf(_P('../README.md')) is None
    # `sanskrit_prose` is the one profile nothing can detect, so `profile=` is the only way in —
    # and it was documented before `add_file` had the argument to honour it
    from litesearch import database as _database
    _pdb = _database(':memory:', sem_search=False)
    _pr = _pdb.add_file(_F/'isopanisad_excerpt.htm', profile='sanskrit_prose')
    assert _pr['kind'] == 'sanskrit' and _pr['chunks'], _pr

In [ ]:
#| hide
# The lemma seam is opt-in and must be a no-op without a pipeline: same object, same metadata.
assert sanskrit_meta(None) is verse_meta
assert lemma_facets('rāmo gacchati', None) == {}
from litesearch.data import profile_for as _pf2
register_profiles()
assert _pf2(name='sanskrit_verse').meta is verse_meta

# With one, metre and lemmas compose into the single `metadata` dict, and only lemmas that differ
# from the surface are kept — the surface is already indexed in `content`.
from litesearch.graph import spacy_pipe as _sp
_en = _sp()          # English stands in: the seam is language-agnostic, the model is not
if _en:
    _f = lemma_facets('The children were running and the mice ate the cheeses.', _en)
    assert 'lemma' in _f and 'child' in _f['lemma'] and 'mouse' in _f['lemma'], _f
    assert 'the' not in _f['lemma'].split()                  # lemma == surface, nothing to add
    _m = sanskrit_meta(_en)
    _v2 = 'dharmakṣetre kurukṣetre samavetā yuyutsavaḥ | māmakāḥ pāṇḍavāś caiva kim akurvata saṃjaya ||'
    assert _m(_v2)['meter'] == 'anuṣṭubh'                    # metre survives the composition
    register_profiles(nlp=_en)
    assert _pf2(name='sanskrit_verse').meta is not verse_meta
    assert _pf2(name='sanskrit_prose').meta is _pf2(name='sanskrit_verse').meta
    register_profiles()                                      # leave the default registration alone
assert _pf2(name='sanskrit_verse').meta is verse_meta


### End to end

The payoff: `add_file` on a Sanskrit source now produces a real tree, verse-sized chunks and
breadcrumbs that name the division a verse sits in — with no arguments and no caller changes.

In [ ]:
#| hide
import numpy as np
from functools import partial
from litesearch import database, hash_embed
from fastcore.all import Path as _P
_F = _P('sanskrit')
if _F.exists():
    _emb = partial(hash_embed, ndim=64, dtype=np.float16)
    _db = database(':memory:', sem_search=False)
    _r = _db.add_file(_F/'manu_tei_excerpt.xml', emb_fn=_emb)
    assert _r['kind'] == 'sanskrit' and _r['chunks'] >= 5, _r
    _toc = _db.toc()[0]['tree']
    assert _toc['children'], 'verse mode must build a real tree, not one flat node'
    _nid = _db.get_tree('store').store(select='node_id', limit=1)[0]['node_id']
    assert '›' in _db.breadcrumb(_nid)          # a hit knows which division it came from

    # and the folded index is live on a real store
    _db2 = database(':memory:', sem_search=False)
    _db2.add_file(_F/'lalita_excerpt.xml', emb_fn=_emb)
    def _n(q): return len(list(_db2.q('select rowid from store_fts where store_fts match ?', (q,))))
    # the excerpt stores `सर॑स्वती` *accented*; all four of these must reach it
    assert _n('सर॑स्वती'), 'exact accented form must match'
    assert _n('सरस्वती'), 'unaccented Devanagari query must reach accented text'
    assert _n('sarasvatī') and _n('sarasvati'), 'IAST and bare ASCII must reach Devanagari'

    # --- metre arrives in the store, and is searchable, with no caller changes ------------------
    import json as _json
    _md = [_json.loads(r['metadata'] or '{}') for r in _db.t.store(select='metadata')]
    # Manu is anuṣṭubh throughout; every verse chunk of it must say so
    assert sum(1 for m in _md if m.get('meter') == 'anuṣṭubh') >= 5, _md
    assert all(m.get('pada') == '8' for m in _md if m.get('meter') == 'anuṣṭubh')
    assert any('pathyā' in (m.get('variant') or '') for m in _md)
    # the citation printed with the verse must not scan: with `|| Manu_1.1 ||` counted, every one
    # of these is 34 syllables and matches nothing
    assert not any(m.get('pada') == '8' and m.get('meter') is None for m in _md)
    # metadata is in the FTS index, so the facet is reachable by ordinary search — in either
    # script, because the folding tokenizer covers the metre name too
    assert _db.t.store.fts_search('anuṣṭubh') and _db.t.store.fts_search('anustubh')
    # and the gaṇa signature survives as ONE token: `_` is a word joiner in this chain
    _g = next(m['gana'].split()[0] for m in _md if m.get('gana'))
    assert '_' in _g and _db.t.store.fts_search(f'"{_g}"'), _g
    assert _db.by_meter(meter='anuṣṭubh') and _db.by_meter(gana=_g)
    assert _db.by_meter() == []                     # no criterion is not "everything"